# PhysicsFormer Stage-1 Pretraining on Colab (thin CLI wrapper)

This notebook is a **thin Colab wrapper** around the canonical Stage-1 entry
script `physics_former/run_physics_training.py` -- the same script that
produced the released `physics_former_best.pt` checkpoint.

It does not reimplement training. It just:
1. mounts Drive, unpacks the snapshot, installs deps,
2. wires `PHYSICS_DATA_DIR` / `CHECKPOINT_DIR` env vars to Drive paths so
   checkpoints persist across Colab sessions, and
3. shells out to `python physics_former/run_physics_training.py --config a100`
   with `--resume <latest>` auto-detected from your Drive checkpoint dir.

## ⚠️  Realistic expectations

Stage-1 takes **3-5 days on a single A100 80GB**. Colab sessions cap at
12-24h, so a full retrain requires multiple sessions. The notebook is
designed for that: re-run all cells in each session and Cell 5 will pick
up the most recent `*.pt` in your Drive checkpoint dir and pass it to
`--resume`. After ~5-10 sessions look for `physics_former_best.pt`.

For paper reproduction in a sane wallclock, run the CLI on local A100
hardware instead -- see `REPRODUCTION.md` §7 Stage-1.

## Prerequisites

1. **Code zip** at `/MyDrive/physics_llm/compsac_2026_code.zip`
   (PowerShell: `Compress-Archive compsac_2026_code\* compsac_2026_code.zip`).
2. **Isaac Sim physics HDF5s** at `/MyDrive/physics_llm/physics_data/`
   (~30 GB; expect slow reads via Drive FUSE -- consider rsync’ing to
   `/content/data/` at session start if Drive throughput is a bottleneck).
3. Runtime → Change runtime type → **A100 GPU** (required for `--config a100`).

Stage-1 *outputs* `physics_former_best.pt`, the Stage-2 adapter notebook
(`colab_train_adapter.ipynb`) *consumes* it.


In [ ]:
# Cell 1: GPU check. A100 80GB is required for --config a100; warn loudly otherwise.
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:  {name}")
    print(f"VRAM: {vram:.1f} GB")
    if 'A100' not in name:
        print(
            f"\n⚠️  --config a100 assumes A100 80GB. {name} will likely OOM."
            "\n   Fall back to --config aggressive in Cell 5 (TRAIN_CONFIG = 'aggressive')."
            "\n   NOTE: aggressive uses a smaller architecture and is NOT byte-compatible"
            "\n   with the released physics_former_best.pt -- use only for sanity testing."
        )


In [ ]:
# Cell 2: Mount Drive and verify required files.
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE        = '/content/drive/MyDrive/physics_llm'
ZIP_PATH          = f'{DRIVE_BASE}/compsac_2026_code.zip'
PHYSICS_DATA_DIR  = f'{DRIVE_BASE}/physics_data'        # Isaac Sim HDF5 directory
CHECKPOINT_DIR    = f'{DRIVE_BASE}/stage1_checkpoints'  # Stage-1 outputs land here

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Checking required inputs on Drive:')
print(f'  code zip:         {"✅" if os.path.exists(ZIP_PATH) else "❌ MISSING"}  {ZIP_PATH}')
print(f'  physics HDF5 dir: {"✅" if os.path.isdir(PHYSICS_DATA_DIR) else "❌ MISSING"}  {PHYSICS_DATA_DIR}')
if os.path.isdir(PHYSICS_DATA_DIR):
    h5_files = [f for f in os.listdir(PHYSICS_DATA_DIR) if f.endswith('.h5')]
    print(f'                    → {len(h5_files)} HDF5 file(s) detected')
    if not h5_files:
        print(f'                    ⚠️  No .h5 files found; Stage-1 will fail at dataloader init.')
print(f'\n  checkpoint output: {CHECKPOINT_DIR}')

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'Missing {ZIP_PATH}; upload compsac_2026_code.zip to Drive first.')
if not os.path.isdir(PHYSICS_DATA_DIR):
    raise FileNotFoundError(
        f'Missing {PHYSICS_DATA_DIR}; stage Isaac Sim physics HDF5 files there '
        f'(see data_generation/isaac_sim/SETUP.md in the snapshot).'
    )


In [ ]:
# Cell 3: Extract the snapshot from Drive (idempotent -- safe to re-run).
%cd /content
!unzip -q -o {ZIP_PATH} -d /content/
print('✅ Snapshot extracted.')

# Auto-detect the extracted code root: walk /content for the dir containing
# physics_former/. Works whether the zip's top-level folder is
# compsac_2026_code/, physics_llm_colab/, or the contents extract flat.
_IGNORED = {'drive', '.config', 'sample_data', '__pycache__', '.git', '.ipynb_checkpoints'}
CODE_ROOT = None
for _root, _dirs, _ in os.walk('/content'):
    _dirs[:] = [d for d in _dirs if d not in _IGNORED]
    if 'physics_former' in _dirs:
        CODE_ROOT = _root
        break

if CODE_ROOT is None:
    raise RuntimeError(
        "Couldn't find physics_former/ under /content. Did Cell 3 actually unzip? "
        "Check the zip layout."
    )
print(f'✅ Code root: {CODE_ROOT}')


In [ ]:
# Cell 4: Install dependencies. (Colab already ships torch/transformers, but
# pinning here makes the wrapper portable to fresh runtimes.)
!pip install -q torch transformers tqdm h5py numpy
print('✅ Dependencies installed.')


In [ ]:
# Cell 5: Wire env vars (read by config_a100.py / config.py) and auto-detect
# the latest Stage-1 checkpoint to resume from.
#
# config.py reads PHYSICS_DATA_DIR and CHECKPOINT_DIR; setting them here means
# the CLI in Cell 6 doesn't need any extra path flags.

import os, glob

# --- training config ---------------------------------------------------------
# 'a100'        → A100Config (768d, 8 layers, 24 heads). Required to land on the
#                 same architecture as the released physics_former_best.pt.
# 'aggressive'  → fallback for non-A100 GPUs. NOT byte-compatible with the
#                 released checkpoint; use only for sanity testing.
TRAIN_CONFIG = 'a100'

os.environ['PHYSICS_DATA_DIR'] = PHYSICS_DATA_DIR
os.environ['CHECKPOINT_DIR']   = CHECKPOINT_DIR

print(f'PHYSICS_DATA_DIR = {os.environ["PHYSICS_DATA_DIR"]}')
print(f'CHECKPOINT_DIR   = {os.environ["CHECKPOINT_DIR"]}')
print(f'TRAIN_CONFIG     = {TRAIN_CONFIG}')

# --- find latest checkpoint for --resume -------------------------------------
# run_physics_training.py writes per-epoch checkpoints under CHECKPOINT_DIR
# (e.g. stage1_epoch{N}.pt) plus stage1_best.pt and physics_former_best.pt.
# Grab the newest *.pt by mtime; the script's own load_checkpoint() will
# restore curriculum, optimizer, and epoch counter.
candidates = sorted(glob.glob(f'{CHECKPOINT_DIR}/*.pt'), key=os.path.getmtime)
if candidates:
    RESUME_PATH = candidates[-1]
    size_mb = os.path.getsize(RESUME_PATH) / 1e6
    print(f'\n✅ Resuming from {os.path.basename(RESUME_PATH)} ({size_mb:.1f} MB)')
else:
    RESUME_PATH = None
    print('\n[fresh run] No prior checkpoint -- Stage-1 will start from scratch.')


In [ ]:
# Cell 6: Shell out to the canonical Stage-1 CLI training script.
#
# This is the SAME command from REPRODUCTION.md §7 -- the wrapper just sets
# env vars (Cell 5), plumbs --resume for multi-session continuity, and tees
# stdout to a Drive-persisted log so progress survives session disconnects.

import subprocess, shlex, datetime

cmd = [
    'python', os.path.join(CODE_ROOT, 'physics_former', 'run_physics_training.py'),
    '--config', TRAIN_CONFIG,
    '--checkpoint-dir', CHECKPOINT_DIR,
]
if RESUME_PATH:
    cmd += ['--resume', RESUME_PATH]

log_path = f'{CHECKPOINT_DIR}/training.log'
print(f'Command: {" ".join(shlex.quote(c) for c in cmd)}')
print(f'Log:     {log_path} (tee’d to this cell)\n')

with open(log_path, 'a', buffering=1) as logf:
    logf.write(f'\n=== Session start: {datetime.datetime.now().isoformat()} ===\n')
    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end='')
        logf.write(line)
    p.wait()
    logf.write(f'=== Session end (exit {p.returncode}): '
               f'{datetime.datetime.now().isoformat()} ===\n')

print(f'\n=== Stage-1 process exited with code {p.returncode} ===')
if p.returncode != 0:
    print('⚠️  Non-zero exit. If Colab disconnected mid-training the latest checkpoint')
    print('    is still on Drive -- just re-run all cells next session and Cell 5 will resume.')


In [ ]:
# Cell 7: Inventory of saved Stage-1 checkpoints + verify physics_former_best.pt.
import glob, os

checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/*.pt'))
if not checkpoints:
    print('⚠️  No checkpoints saved -- Stage-1 likely failed before its first save.')
    print(f'    Inspect {CHECKPOINT_DIR}/training.log for the traceback.')
else:
    print(f'Saved Stage-1 checkpoints in {CHECKPOINT_DIR}:')
    for cp in checkpoints:
        size_mb = os.path.getsize(cp) / 1e6
        print(f'  {os.path.basename(cp):<40} {size_mb:>8.1f} MB')

best = f'{CHECKPOINT_DIR}/physics_former_best.pt'
if os.path.exists(best):
    print(
        '\n✅ physics_former_best.pt produced. Stage-1 reached the paper-equivalent'
        '\n   checkpoint (modulo data-shuffle / CUDA non-determinism). Drop it into'
        '\n   compsac_2026_code/checkpoints/ and proceed to Stage-2 via colab_train_adapter.ipynb.'
    )
else:
    print(
        '\n[in progress] physics_former_best.pt not yet produced (curriculum still'
        '\n              advancing). Re-run Cells 1-7 in your next Colab session;'
        '\n              Cell 5 picks up the latest checkpoint automatically.'
    )
